In [ ]:
# Cell 1: reinstall pinned versions (known working combo)
!pip install "datasets<4.0.0" -q
!pip install transformers==4.46.0 huggingface_hub==0.26.0 tokenizers==0.20.3 safetensors==0.4.5 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.2 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.4/447.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB

In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/train_processed.csv')
val_df = pd.read_csv('/content/val_processed.csv')
test_df = pd.read_csv('/content/test_processed.csv')

print(train_df.shape, val_df.shape, test_df.shape)

(15383, 21) (1922, 19) (1924, 19)


In [ ]:
# Reapply the empty-row drop (CSV was saved before this step happened)
train_df = train_df[train_df['clean_text'].str.strip() != ''].reset_index(drop=True)
print(train_df.shape)  # should now show 15382

(15383, 21)


In [ ]:
target_cols = [c for c in train_df.columns if c.startswith('target_')]

y_train_label = train_df['final_label']
y_val_label = val_df['final_label']
y_test_label = test_df['final_label']

y_train_targets = train_df[target_cols]
y_val_targets = val_df[target_cols]
y_test_targets = test_df[target_cols]

print(y_train_targets.shape, y_val_targets.shape)

(15383, 13) (1922, 13)


In [ ]:
empty_check = (train_df['clean_text'].str.strip() == '').sum()
print("Empty clean_text rows remaining:", empty_check)

Empty clean_text rows remaining: 0


In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertModel
import torch.nn as nn
from sklearn.metrics import classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [ ]:
print("NaNs in train clean_text:", train_df['clean_text'].isna().sum())
print("NaNs in val clean_text:", val_df['clean_text'].isna().sum())
print("NaNs in test clean_text:", test_df['clean_text'].isna().sum())

NaNs in train clean_text: 1
NaNs in val clean_text: 0
NaNs in test clean_text: 0


In [ ]:
# Replace any NaN values with an empty string, then re-check
train_df['clean_text'] = train_df['clean_text'].fillna('')
val_df['clean_text'] = val_df['clean_text'].fillna('')
test_df['clean_text'] = test_df['clean_text'].fillna('')

print("NaNs after fix:", train_df['clean_text'].isna().sum(), val_df['clean_text'].isna().sum(), test_df['clean_text'].isna().sum())

NaNs after fix: 0 0 0


In [ ]:
MAX_LEN_BERT = 50

distil_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def encode_texts(texts, tokenizer, max_len=MAX_LEN_BERT):
    return tokenizer(
        list(texts),
        truncation=True,
        padding='max_length',
        max_length=max_len,
        return_tensors='pt'
    )

train_encodings = encode_texts(train_df['clean_text'], distil_tokenizer)
val_encodings = encode_texts(val_df['clean_text'], distil_tokenizer)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, encodings, toxicity_labels, target_labels):
        self.encodings = encodings
        self.toxicity_labels = toxicity_labels
        self.target_labels = target_labels
    def __len__(self):
        return len(self.toxicity_labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['toxicity_label'] = torch.tensor(self.toxicity_labels[idx], dtype=torch.long)
        item['target_label'] = torch.tensor(self.target_labels[idx], dtype=torch.float)
        return item

train_dataset = ToxicityDataset(train_encodings, y_train_label.values, y_train_targets.values)
val_dataset = ToxicityDataset(val_encodings, y_val_label.values, y_val_targets.values)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
class DistilBertMultiTask(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.toxicity_head = nn.Linear(768, 3)
        self.target_head = nn.Linear(768, 13)
    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = output.last_hidden_state[:, 0, :]
        x = self.dropout(cls_token)
        return self.toxicity_head(x), self.target_head(x)

distil_model = DistilBertMultiTask().to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
optimizer = AdamW(distil_model.parameters(), lr=2e-5)
toxicity_loss_fn = nn.CrossEntropyLoss()
target_loss_fn = nn.BCEWithLogitsLoss()
EPOCHS = 3

for epoch in range(EPOCHS):
    distil_model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        toxicity_labels = batch['toxicity_label'].to(device)
        target_labels = batch['target_label'].to(device)

        toxicity_logits, target_logits = distil_model(input_ids, attention_mask)
        loss = toxicity_loss_fn(toxicity_logits, toxicity_labels) + target_loss_fn(target_logits, target_labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/3 - Avg Loss: 1.1205
Epoch 2/3 - Avg Loss: 0.9078
Epoch 3/3 - Avg Loss: 0.7478


In [ ]:
distil_model.eval()
all_toxicity_preds = []
all_target_preds = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        toxicity_logits, target_logits = distil_model(input_ids, attention_mask)
        toxicity_preds = torch.argmax(toxicity_logits, dim=1).cpu().numpy()
        target_preds = (torch.sigmoid(target_logits) > 0.5).int().cpu().numpy()
        all_toxicity_preds.extend(toxicity_preds)
        all_target_preds.extend(target_preds)

all_target_preds = np.array(all_target_preds)

print("=== Toxicity Classification (DistilBERT) ===")
print(classification_report(y_val_label, all_toxicity_preds, target_names=['hatespeech','normal','offensive']))

print("=== Target Community Classification (DistilBERT) ===")
print(classification_report(y_val_targets, all_target_preds, target_names=target_cols, zero_division=0))

=== Toxicity Classification (DistilBERT) ===
              precision    recall  f1-score   support

  hatespeech       0.74      0.79      0.77       593
      normal       0.72      0.71      0.72       781
   offensive       0.54      0.51      0.53       548

    accuracy                           0.68      1922
   macro avg       0.67      0.67      0.67      1922
weighted avg       0.68      0.68      0.68      1922

=== Target Community Classification (DistilBERT) ===
                       precision    recall  f1-score   support

       target_African       0.83      0.74      0.78       419
         target_Islam       0.78      0.81      0.80       299
         target_Women       0.72      0.36      0.48       383
        target_Jewish       0.96      0.73      0.83       241
    target_Homosexual       0.83      0.74      0.78       251
         target_Other       0.64      0.22      0.33       366
       target_Refugee       0.79      0.49      0.60       157
          target

In [ ]:
# Get DistilBERT's test-set predictions now, while the model is still in memory
test_encodings_distil = encode_texts(test_df['clean_text'], distil_tokenizer)
test_dataset_distil = ToxicityDataset(test_encodings_distil, y_test_label.values, y_test_targets.values)
test_loader_distil = DataLoader(test_dataset_distil, batch_size=16)

distil_model.eval()
distil_test_tox_preds, distil_test_tgt_preds = [], []

with torch.no_grad():
    for batch in test_loader_distil:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        toxicity_logits, target_logits = distil_model(input_ids, attention_mask)
        distil_test_tox_preds.extend(torch.argmax(toxicity_logits, dim=1).cpu().numpy())
        distil_test_tgt_preds.extend((torch.sigmoid(target_logits) > 0.5).int().cpu().numpy())

distil_test_tgt_preds = np.array(distil_test_tgt_preds)

from sklearn.metrics import accuracy_score, f1_score

distilbert_test_results = {
    'toxicity_acc': accuracy_score(y_test_label, distil_test_tox_preds),
    'toxicity_macroF1': f1_score(y_test_label, distil_test_tox_preds, average='macro'),
    'target_microF1': f1_score(y_test_targets, distil_test_tgt_preds, average='micro'),
    'target_macroF1': f1_score(y_test_targets, distil_test_tgt_preds, average='macro')
}

print(distilbert_test_results)

{'toxicity_acc': 0.6808731808731808, 'toxicity_macroF1': 0.6672581970937306, 'target_microF1': 0.6123388581952118, 'target_macroF1': 0.4811688659563122}


In [ ]:
del distil_model, test_loader_distil
torch.cuda.empty_cache()
import gc
gc.collect()

0

In [ ]:
from transformers import BertTokenizerFast, BertModel

bert_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

train_encodings_bert = encode_texts(train_df['clean_text'], bert_tokenizer)
val_encodings_bert = encode_texts(val_df['clean_text'], bert_tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
train_dataset_bert = ToxicityDataset(train_encodings_bert, y_train_label.values, y_train_targets.values)
val_dataset_bert = ToxicityDataset(val_encodings_bert, y_val_label.values, y_val_targets.values)

train_loader_bert = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
val_loader_bert = DataLoader(val_dataset_bert, batch_size=16)

In [ ]:
class BertMultiTask(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.toxicity_head = nn.Linear(768, 3)
        self.target_head = nn.Linear(768, 13)
    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = output.last_hidden_state[:, 0, :]
        x = self.dropout(cls_token)
        return self.toxicity_head(x), self.target_head(x)

bert_model = BertMultiTask().to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
optimizer_bert = AdamW(bert_model.parameters(), lr=2e-5)

for epoch in range(EPOCHS):
    bert_model.train()
    total_loss = 0
    for batch in train_loader_bert:
        optimizer_bert.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        toxicity_labels = batch['toxicity_label'].to(device)
        target_labels = batch['target_label'].to(device)

        toxicity_logits, target_logits = bert_model(input_ids, attention_mask)
        loss = toxicity_loss_fn(toxicity_logits, toxicity_labels) + target_loss_fn(target_logits, target_labels)

        loss.backward()
        optimizer_bert.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {total_loss/len(train_loader_bert):.4f}")

Epoch 1/3 - Avg Loss: 1.1282
Epoch 2/3 - Avg Loss: 0.9080
Epoch 3/3 - Avg Loss: 0.7374


In [ ]:
bert_model.eval()
all_toxicity_preds_bert = []
all_target_preds_bert = []

with torch.no_grad():
    for batch in val_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        toxicity_logits, target_logits = bert_model(input_ids, attention_mask)
        all_toxicity_preds_bert.extend(torch.argmax(toxicity_logits, dim=1).cpu().numpy())
        all_target_preds_bert.extend((torch.sigmoid(target_logits) > 0.5).int().cpu().numpy())

all_target_preds_bert = np.array(all_target_preds_bert)

print("=== Toxicity Classification (BERT) ===")
print(classification_report(y_val_label, all_toxicity_preds_bert, target_names=['hatespeech','normal','offensive']))

print("=== Target Community Classification (BERT) ===")
print(classification_report(y_val_targets, all_target_preds_bert, target_names=target_cols, zero_division=0))

=== Toxicity Classification (BERT) ===
              precision    recall  f1-score   support

  hatespeech       0.72      0.79      0.75       593
      normal       0.72      0.73      0.72       781
   offensive       0.54      0.47      0.51       548

    accuracy                           0.68      1922
   macro avg       0.66      0.66      0.66      1922
weighted avg       0.67      0.68      0.67      1922

=== Target Community Classification (BERT) ===
                       precision    recall  f1-score   support

       target_African       0.81      0.78      0.79       419
         target_Islam       0.82      0.78      0.80       299
         target_Women       0.67      0.42      0.51       383
        target_Jewish       0.95      0.84      0.89       241
    target_Homosexual       0.83      0.73      0.78       251
         target_Other       0.66      0.19      0.29       366
       target_Refugee       0.82      0.45      0.58       157
          target_Arab       

In [ ]:
test_encodings_bert = encode_texts(test_df['clean_text'], bert_tokenizer)
test_dataset_bert = ToxicityDataset(test_encodings_bert, y_test_label.values, y_test_targets.values)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=16)

bert_model.eval()
bert_test_tox_preds, bert_test_tgt_preds = [], []

with torch.no_grad():
    for batch in test_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        toxicity_logits, target_logits = bert_model(input_ids, attention_mask)
        bert_test_tox_preds.extend(torch.argmax(toxicity_logits, dim=1).cpu().numpy())
        bert_test_tgt_preds.extend((torch.sigmoid(target_logits) > 0.5).int().cpu().numpy())

bert_test_tgt_preds = np.array(bert_test_tgt_preds)

bert_test_results = {
    'toxicity_acc': accuracy_score(y_test_label, bert_test_tox_preds),
    'toxicity_macroF1': f1_score(y_test_label, bert_test_tox_preds, average='macro'),
    'target_microF1': f1_score(y_test_targets, bert_test_tgt_preds, average='micro'),
    'target_macroF1': f1_score(y_test_targets, bert_test_tgt_preds, average='macro')
}

print(bert_test_results)

{'toxicity_acc': 0.6746361746361746, 'toxicity_macroF1': 0.6578968262285664, 'target_microF1': 0.613424345847554, 'target_macroF1': 0.45694411828055337}
